# Anotador clínico TDAH · BRIEF-2 con evidencia por ítem

Este cuaderno convierte la **nota espontánea de un cuidador** (texto libre, escrita cuando él quiera) en una **anotación clínica estructurada** según el instrumento BRIEF-2, en la que **cada ítem detectado va acompañado de su evidencia**: la cita textual de la nota que lo sustenta y una justificación de por qué corresponde a ese ítem.

La evidencia por ítem tiene dos funciones:
1. **Auditabilidad clínica** — el médico puede verificar cada ítem contra el fragmento exacto de la nota, sin releerla entera.
2. **Verificación automática** — como la evidencia debe ser una cita literal, el cuaderno comprueba si ese fragmento existe realmente en la nota. La métrica `evidencia_ok` detecta alucinaciones sin necesidad de revisión humana.

**Backend**: salida estructurada con LangChain (`json_schema`), que impone el esquema en la decodificación. La comparación empírica que justifica esta elección (cuatro backends evaluados) está en la rama `fase1-backends`.

**Cómo usarlo**: ajustar la celda de parámetros y ejecutar de arriba abajo. Cada ejecución guarda sus resultados en la tabla `experimento` con un código, comparable en `comparacion_experimentos.ipynb`.

## 1 · Parámetros

In [ ]:
import datetime as dt
import json
import sqlite3
import time
import unicodedata

import pandas as pd

# --- Parámetros del experimento (lo único que hay que tocar) ---
NOTAS        = None         # None = todas las notas; o lista de id_entrada: [5, 12, 30]
PACIENTES    = None         # None = todos; o lista: ["PAC001", "PAC003"]
MUESTRA      = 10           # nº máximo de notas a anotar (None = sin límite)
REPETICIONES = 3            # veces que se anota cada nota
TEMPERATURA  = 0.7
MODELO       = "gemma4:e4b" # en Mercurio: gemma4:26b

# --- Rutas y conexión ---
RUTA_BD     = "datos/anotador.db"
OLLAMA_URL  = "http://127.0.0.1:11002"   # puerto del túnel a Ollama
INSTRUMENTO = "instrumentos/brief2.json"

BACKEND     = "langchain"
EXPERIMENTO = f"anotador-evidencia-t{TEMPERATURA}-{dt.date.today():%Y%m%d}"
print(f"Código de experimento: {EXPERIMENTO}")

## 2 · Datos: las notas de los cuidadores

La unidad de trabajo es **la nota**: los cuidadores escriben cuando quieren, así que un paciente puede tener varias notas una semana y ninguna la siguiente. Cada nota se anota de forma independiente.

> Limitación conocida a tener en cuenta en el análisis: las notas espontáneas tienden a escribirse cuando algo va mal (sesgo de selección de eventos). La ausencia de notas no implica estabilidad clínica.

In [ ]:
instrumento = json.load(open(INSTRUMENTO, encoding="utf-8"))
print(f"Instrumento: {instrumento['nombre']} ({len(instrumento['items'])} ítems)")

con = sqlite3.connect(RUTA_BD)

notas = pd.read_sql(
    '''
    SELECT e.id_entrada, e.id_paciente, c.rol AS informante, e.fecha,
           p.fecha_nacimiento, p.sexo, e.texto
    FROM entrada e
    JOIN paciente p USING (id_paciente)
    JOIN cuidador c ON c.id_cuidador = e.id_cuidador
    ORDER BY e.id_paciente, e.fecha
    ''',
    con,
)
if NOTAS:
    notas = notas[notas["id_entrada"].isin(NOTAS)]
if PACIENTES:
    notas = notas[notas["id_paciente"].isin(PACIENTES)]
if MUESTRA:
    notas = notas.head(MUESTRA)


def calcular_edad(nacimiento, observacion):
    nac = pd.to_datetime(nacimiento).date()
    obs = pd.to_datetime(observacion).date()
    return obs.year - nac.year - ((obs.month, obs.day) < (nac.month, nac.day))


notas["edad"] = [
    calcular_edad(n, f) for n, f in zip(notas["fecha_nacimiento"], notas["fecha"])
]

print(f"{len(notas)} notas de {notas['id_paciente'].nunique()} pacientes")
notas[["id_entrada", "id_paciente", "informante", "fecha", "edad", "texto"]].head()

## 3 · El esquema de salida: evidencia por ítem

El contrato Pydantic que el modelo debe rellenar. Cada ítem es un objeto `{id, evidencia, justificacion}` — no un número suelto.

In [ ]:
from pydantic import BaseModel, Field


class ItemDetectado(BaseModel):
    id: int = Field(description="Número del ítem del catálogo BRIEF-2")
    evidencia: str = Field(
        description="Cita TEXTUAL de la nota, copiada literalmente, que sustenta el ítem"
    )
    justificacion: str = Field(
        description="Por qué esa evidencia corresponde a este ítem (una frase)"
    )


class Anotacion(BaseModel):
    items: list[ItemDetectado] = Field(default_factory=list)
    escalas_afectadas: list[str] = Field(default_factory=list)
    nivel_alerta: str = "bajo"
    nota_clinica: str = ""

## 4 · Prompts

El prompt de sistema se construye desde `brief2.json` e instruye al modelo para citar **literalmente** la nota en cada evidencia — y para NO incluir un ítem si no puede citarlo.

In [ ]:
COMILLAS = '"' * 3  # delimitador del texto de la nota dentro del prompt


def construir_prompt_sistema(instrumento):
    catalogo = "\n".join(
        f"  {it['id']}: [{it['escala']}] {it['texto']}" for it in instrumento["items"]
    )
    escalas = "\n".join(f"  - {e}: {d}" for e, d in instrumento["escalas"].items())
    n = instrumento["niveles_alerta"]
    return f'''Eres un {instrumento["rol_anotador"]}.

Tu tarea es analizar la nota libre de un cuidador sobre su hijo/a y producir una
anotación clínica estructurada basada en el instrumento {instrumento["nombre"]}.

## CATÁLOGO DE ÍTEMS ({len(instrumento["items"])} ítems)
{catalogo}

## ESCALAS
{escalas}

## NIVELES DE ALERTA
{" | ".join(n)}

## INSTRUCCIONES DE SALIDA
Para CADA ítem que detectes debes aportar:
- "id": el número del ítem del catálogo.
- "evidencia": la cita TEXTUAL de la nota que lo sustenta, copiada literalmente,
  sin parafrasear ni corregir. Si no puedes citar un fragmento literal que lo
  sustente, NO incluyas el ítem.
- "justificacion": una frase explicando por qué esa evidencia corresponde al ítem.

Incluye solo ítems observables en la nota. Completa además "escalas_afectadas"
(las escalas de los ítems detectados), "nivel_alerta" ({n[0]}|{n[1]}|{n[2]}) y
"nota_clinica" (resumen de 1-3 frases para el médico).'''


def construir_prompt_usuario(e):
    return f'''## CONTEXTO DEL PACIENTE
- Edad: {e.edad} años
- Sexo: {e.sexo}
- Informante: {e.informante}
- Fecha de la nota: {e.fecha}

## NOTA DEL CUIDADOR
{COMILLAS}{e.texto}{COMILLAS}

Analiza la nota y genera la anotación clínica estructurada.'''


prompt_sistema = construir_prompt_sistema(instrumento)
print(prompt_sistema[:400] + "\n[...]")

## 5 · El backend (LangChain · json_schema)

El esquema JSON derivado de `Anotacion` se impone al modelo en el momento de la decodificación: la salida no puede salirse de la estructura.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=MODELO,
    base_url=OLLAMA_URL,
    temperature=TEMPERATURA,
    num_predict=2048,
)
llm_estructurado = llm.with_structured_output(Anotacion, method="json_schema")


def anotar(prompt_sistema, prompt_usuario):
    '''Llama al modelo y devuelve (anotacion | None, respuesta_cruda).'''
    try:
        obj = llm_estructurado.invoke(
            [SystemMessage(prompt_sistema), HumanMessage(prompt_usuario)]
        )
    except Exception as e:
        return None, f"error: {e}"
    if obj is None:
        return None, ""
    datos = obj if isinstance(obj, dict) else obj.model_dump()
    return datos, json.dumps(datos, ensure_ascii=False)

## 6 · Verificación automática de la evidencia

Comprueba si cada evidencia aparece **literalmente** en la nota (tras normalizar mayúsculas, tildes y espacios). Una evidencia que no aparece es una señal de alucinación: el modelo atribuyó al cuidador algo que no escribió.

In [ ]:
def _normalizar(s):
    '''Minúsculas, sin tildes y espacios colapsados: tolera diferencias
    triviales de copia sin dejar de exigir una cita literal.'''
    s = unicodedata.normalize("NFKD", str(s).lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return " ".join(s.split())


def verificar_evidencias(anotacion, texto_nota):
    '''Fracción de ítems cuya evidencia aparece literalmente en la nota.

    Devuelve (fraccion, detalle) donde detalle es [(id_item, True/False), ...].
    Si la anotación no tiene ítems, devuelve (None, []).
    '''
    items = (anotacion or {}).get("items", [])
    if not items:
        return None, []
    t = _normalizar(texto_nota)
    detalle = [
        (it.get("id"), _normalizar(it.get("evidencia", "")) in t) for it in items
    ]
    fraccion = sum(ok for _, ok in detalle) / len(detalle)
    return fraccion, detalle

## 7 · Una anotación de ejemplo

Una sola nota, para ver la anotación final con sus evidencias verificadas (`OK` = la cita existe en la nota, `??` = no se encuentra) antes de lanzar el experimento.

In [ ]:
ejemplo = notas.iloc[0]
print(f"Nota {ejemplo.id_entrada} · {ejemplo.id_paciente} · {ejemplo.informante} · {ejemplo.fecha}")
print(f"Texto: {ejemplo.texto[:250]}...\n")

t0 = time.time()
anotacion, cruda = anotar(prompt_sistema, construir_prompt_usuario(ejemplo))
print(f"Latencia: {time.time() - t0:.1f}s\n")

if anotacion is None:
    print("[FALLO DE FORMATO] El modelo no devolvió una anotación válida:")
    print(cruda[:500])
else:
    fraccion, detalle = verificar_evidencias(anotacion, ejemplo.texto)
    verificado = dict(detalle)
    print(f"Nivel de alerta : {anotacion['nivel_alerta']}")
    print(f"Escalas         : {anotacion['escalas_afectadas']}")
    if fraccion is None:
        print("Sin ítems detectados")
    else:
        print(f"Evidencia verificada: {fraccion:.0%}")
    print()
    for it in anotacion["items"]:
        marca = "OK " if verificado.get(it["id"]) else "?? "
        print(f"  [{marca}] ítem {it['id']}: \"{it['evidencia']}\"")
        print(f"        → {it['justificacion']}")
    print(f"\nNota clínica: {anotacion['nota_clinica']}")

## 8 · El experimento

Anota cada nota `REPETICIONES` veces y guarda cada resultado en la tabla `experimento`. Además de las columnas habituales se guardan `items_detalle` (los ítems con su evidencia y justificación) y `evidencia_ok` (fracción de evidencias verificadas).

> Si relanzas con el mismo código se añaden filas al mismo experimento. Para empezar de cero: `con.execute("DELETE FROM experimento WHERE codigo = ?", [EXPERIMENTO]); con.commit()`

In [ ]:
con.execute('''
CREATE TABLE IF NOT EXISTS experimento (
    id                INTEGER PRIMARY KEY,
    codigo            TEXT NOT NULL,
    creada_en         TEXT NOT NULL,
    backend           TEXT NOT NULL,
    modelo            TEXT NOT NULL,
    temperatura       REAL NOT NULL,
    semana            INTEGER,
    id_paciente       TEXT,
    id_entrada        INTEGER,
    repeticion        INTEGER,
    formato_ok        INTEGER,
    items_detectados  TEXT,               -- JSON: [int]
    escalas_afectadas TEXT,               -- JSON: [str]
    nivel_alerta      TEXT,
    nota_clinica      TEXT,
    justificacion     TEXT,
    latencia_s        REAL,
    respuesta_cruda   TEXT,               -- salida bruta del modelo (auditoría)
    items_detalle     TEXT,               -- JSON: [{id, evidencia, justificacion}]
    evidencia_ok      REAL                -- fracción de ítems con evidencia literal
)''')
# Si la tabla existe de experimentos anteriores, añade las columnas nuevas
for col, tipo in [("respuesta_cruda", "TEXT"), ("items_detalle", "TEXT"),
                  ("evidencia_ok", "REAL")]:
    try:
        con.execute(f"ALTER TABLE experimento ADD COLUMN {col} {tipo}")
    except sqlite3.OperationalError:
        pass  # la columna ya existe
con.commit()

total = len(notas) * REPETICIONES
print(f"Experimento '{EXPERIMENTO}': {len(notas)} notas × {REPETICIONES} repeticiones "
      f"= {total} llamadas al modelo")

hechas = 0
for _, e in notas.iterrows():
    prompt_usuario = construir_prompt_usuario(e)
    for rep in range(REPETICIONES):
        t0 = time.time()
        anotacion, cruda = anotar(prompt_sistema, prompt_usuario)
        latencia = time.time() - t0
        ok = anotacion is not None
        a = anotacion or {}
        items = a.get("items", [])
        fraccion, _ = verificar_evidencias(a, e.texto) if ok else (None, [])
        con.execute(
            "INSERT INTO experimento (codigo, creada_en, backend, modelo, temperatura, "
            "semana, id_paciente, id_entrada, repeticion, formato_ok, items_detectados, "
            "escalas_afectadas, nivel_alerta, nota_clinica, justificacion, latencia_s, "
            "respuesta_cruda, items_detalle, evidencia_ok) "
            "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (EXPERIMENTO, dt.datetime.now().isoformat(timespec="seconds"), BACKEND,
             MODELO, TEMPERATURA, None, e.id_paciente, int(e.id_entrada), rep,
             int(ok), json.dumps([it.get("id") for it in items]),
             json.dumps(a.get("escalas_afectadas", [])), a.get("nivel_alerta"),
             a.get("nota_clinica"), None, latencia, cruda,
             json.dumps(items, ensure_ascii=False), fraccion),
        )
        con.commit()
        hechas += 1
        if not ok:
            estado = "FALLO DE FORMATO"
        elif fraccion is None:
            estado = "ok (sin ítems)"
        else:
            estado = f"ok · evidencia {fraccion:.0%}"
        print(f"  [{hechas:>3}/{total}] nota {e.id_entrada} ({e.id_paciente}) "
              f"rep {rep + 1} → {estado} ({latencia:.1f}s)")

print(f"\nGuardado en la tabla `experimento` con codigo = '{EXPERIMENTO}'")

## 9 · Resultados

- `formato_ok`: fracción de salidas con la estructura válida.
- `acuerdo_nivel` (0–1): fracción de repeticiones que coincide con el nivel de alerta más frecuente de la nota.
- `evidencia_media` (0–1): fracción de ítems cuya evidencia existe literalmente en la nota. **La métrica nueva de esta fase.**
- `latencia_media`: segundos por anotación.

In [ ]:
df = pd.read_sql(
    "SELECT * FROM experimento WHERE codigo = ?", con, params=[EXPERIMENTO]
)
print(f"{len(df)} anotaciones del experimento '{EXPERIMENTO}'\n")


def acuerdo_modal(niveles):
    '''Fracción de repeticiones que coincide con el nivel más frecuente.'''
    s = niveles.dropna()
    return round(s.value_counts().iloc[0] / len(s), 2) if len(s) else None


resumen = df.groupby("id_entrada").agg(
    id_paciente=("id_paciente", "first"),
    repeticiones=("repeticion", "count"),
    formato_ok=("formato_ok", "mean"),
    acuerdo_nivel=("nivel_alerta", acuerdo_modal),
    evidencia_media=("evidencia_ok", "mean"),
    latencia_media=("latencia_s", "mean"),
).round(2)

print(f"Formato válido                        : {df['formato_ok'].mean():.0%}")
print(f"Acuerdo del nivel entre repeticiones  : {resumen['acuerdo_nivel'].mean():.2f}")
print(f"Evidencia verificada (media)          : {df['evidencia_ok'].mean():.0%}")
print(f"Latencia media por anotación          : {df['latencia_s'].mean():.1f}s\n")
resumen

In [ ]:
# Las repeticiones de una nota concreta, ítem a ítem con su evidencia
UNA_NOTA = int(df["id_entrada"].iloc[0])   # cambiar por la nota que interese

d = df[df["id_entrada"] == UNA_NOTA]
texto_nota = notas.loc[notas["id_entrada"] == UNA_NOTA, "texto"].iloc[0]
print(f"Nota {UNA_NOTA}: {texto_nota[:200]}...\n")

for _, fila in d.iterrows():
    ev = f"{fila.evidencia_ok:.0%}" if pd.notna(fila.evidencia_ok) else "—"
    print(f"— repetición {fila.repeticion}: nivel={fila.nivel_alerta} · evidencia {ev}")
    for it in json.loads(fila.items_detalle or "[]"):
        print(f"    ítem {it['id']}: \"{it['evidencia'][:80]}\"")
    print()

## 10 · Siguiente paso

- Comparar experimentos (temperaturas, modelos, versiones del prompt) por su código en `comparacion_experimentos.ipynb`.
- Los pares (ítem, evidencia) con `evidencia_ok` verificada son la entrada natural para la **validación con clínicos**: juzgar si la evidencia sustenta el ítem es mucho más rápido que revisar anotaciones completas.